# Chapitre 18 - Exercices 18.1 à 18.5

- 18.1 : ridge très régularisée et PLS à une composante ;
- 18.2 : centroïdes réduits et lasso ;
- 18.3 : contrainte de somme nulle dans la multinomiale ridge ;
- 18.4 : formule de calcul (18.15) de la ridge ;
- 18.5 : le théorème de réduction (18.16)-(18.17).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl import ch18_grande_dimension as g

## Exercice 18.1 : Ridge pour $\lambda\to\infty$ et PLS à une composante

**Énoncé (reformulé).** Pour un vecteur de coefficients $\hat\beta$, on note
$\hat\beta/\|\hat\beta\|_2$ sa version normalisée. Montrer que, lorsque
$\lambda\to\infty$, les coefficients ridge normalisés convergent vers les
coefficients normalisés de la régression PLS (moindres carrés partiels) à **une
composante**.

**Rappels.**

- Ridge : $\hat\beta_\lambda = (\mathbf X^T\mathbf X+\lambda I)^{-1}\mathbf X^Ty$.
- PLS (algorithme 3.3 du livre), avec des colonnes $x_j$ standardisées et $y$
  centré : la première direction a pour poids $\hat\varphi_{1j} = \langle x_j, y\rangle$,
  la première composante est $z_1 = \sum_j\hat\varphi_{1j}x_j = \mathbf X\mathbf X^Ty$,
  et l'on régresse $y$ sur $z_1$ : $\hat\theta_1 = \langle z_1,y\rangle/\langle z_1,z_1\rangle$.
  L'ajustement à une composante est $\hat y = \hat\theta_1 z_1 = \mathbf X(\hat\theta_1\mathbf X^Ty)$,
  donc les coefficients (dans l'espace des $x_j$) sont
  $$\hat\beta^{\mathrm{PLS}} = \hat\theta_1\,\mathbf X^Ty .$$

**Solution.**

*Coefficients PLS normalisés.* On a
$\hat\theta_1 = \frac{y^T\mathbf X\mathbf X^Ty}{\|z_1\|^2} = \frac{\|\mathbf X^Ty\|^2}{\|z_1\|^2}\ge 0$.
Si $\mathbf X^Ty\neq 0$, alors $\hat\theta_1 > 0$ et
$$\frac{\hat\beta^{\mathrm{PLS}}}{\|\hat\beta^{\mathrm{PLS}}\|} = \frac{\mathbf X^Ty}{\|\mathbf X^Ty\|}.$$

*Ridge pour $\lambda$ grand.* On factorise $\lambda$ :
$$\lambda\hat\beta_\lambda = \lambda(\mathbf X^T\mathbf X + \lambda I)^{-1}\mathbf X^Ty = \Big(I + \tfrac1\lambda\mathbf X^T\mathbf X\Big)^{-1}\mathbf X^Ty .$$
Quand $\lambda\to\infty$, $I + \frac1\lambda\mathbf X^T\mathbf X\to I$, et l'inversion
est continue au voisinage de $I$ ; donc $\lambda\hat\beta_\lambda\to\mathbf X^Ty$.
Plus précisément, pour $\lambda > \|\mathbf X^T\mathbf X\|$ (norme d'opérateur), la série
de Neumann donne
$$\lambda\hat\beta_\lambda = \mathbf X^Ty - \frac1\lambda\mathbf X^T\mathbf X\,\mathbf X^Ty + O(\lambda^{-2}).$$

*Passage à la normalisation.* La normalisation est invariante par multiplication par
un scalaire positif : $\hat\beta_\lambda/\|\hat\beta_\lambda\| = \lambda\hat\beta_\lambda/\|\lambda\hat\beta_\lambda\|$.
L'application $v\mapsto v/\|v\|$ étant continue en tout $v\ne 0$, on obtient
$$\frac{\hat\beta_\lambda}{\|\hat\beta_\lambda\|}\ \xrightarrow[\lambda\to\infty]{}\ \frac{\mathbf X^Ty}{\|\mathbf X^Ty\|} = \frac{\hat\beta^{\mathrm{PLS}}}{\|\hat\beta^{\mathrm{PLS}}\|}. \qquad\square$$

> **Intuition.** Quand la pénalité domine, $\mathbf X^T\mathbf X$ devient négligeable
> devant $\lambda I$ : la ridge « oublie » les corrélations entre variables et chaque
> coefficient devient proportionnel à la covariance univariée $\langle x_j, y\rangle$,
> exactement comme la première direction PLS.

**Vérification numérique** sur les données *prostate* (8 variables standardisées) et
sur un cas simulé avec $p = 500 > N = 40$. On trace $1 - \cos(\hat\beta_\lambda, \mathbf X^Ty)$.